# Analyse extraction quality

0. Extract from manual anotation args in process format
1. Builds optimal pairs with sentence-embedding cosine similarity (fast)
2. **Metric vs. Manual depuration** (overall)
    - Precision = TP / Predicted Positives
    
    Where: True Positives (TP) = total_args_kept for each model., Predicted Positives = total_args for each model.

3. **Metric vs. Manual annotation** (overall and by goal):
    - rougeL_f1_avg (Measures recall/precision of overlapping n-grams or longest common subsequence, pairwise with pairs if max similarity grater than 0.75 is a positive and avg.)
    - bleu_avg (Precision-oriented metric on n-grams, pairwise with pairs of max similarity and avg.)
    - Recall@k: proportion of gold arguments retrieved.
    - Precision: proportion of retrieved arguments that match gold.
    - F1-score: balance of precision and recall.
    - Coverage Ratio: (# of distinct reference arguments covered) ÷ (total reference arguments).
    - Overgeneration: (# retrieved – # matched) ÷ (# retrieved).

4.  **Metric vs. Manual annotation** (between models agreement): 
    - Fleiss’ κ across models (How consistently multiple raters assign labels to the same units, so in extraction case since models don’t label the exact same argument units, agreement is mostly chance-level.)
    - Jaccard (Overlap-based set similarity) 
    - Fuzzy-matching Jaccard (cosine/embedding-based instead of exact string)
    - % of gold arguments recovered by ≥1, 2, 3 and all model.
    - % of model arguments supported by another model.




In [ ]:
#%pip install -q sentence-transformers rouge-score bert-score nltk hf_xet


Note: you may need to restart the kernel to use updated packages.


In [1]:
import nltk
nltk.download('punkt', quiet=True)

import re
from nltk.tokenize import TreebankWordTokenizer
_tok = TreebankWordTokenizer()

import os, re, json, math
from typing import List, Dict, Tuple
import numpy as np
import pandas as pd
from collections import defaultdict, OrderedDict

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# BERTScore (computationally expensive)
USE_BERTSCORE = False
if USE_BERTSCORE:
    from bert_score import score as bert_score

c:\Users\cpalo\.conda\envs\pddlgym\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 0. Extract from manual anotation args in process format

In [2]:
import os, re, json
import pandas as pd
from collections import defaultdict

def make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name="TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename=None,
    txt_basename=None,
    keep_duplicates=True
):
    # 1. Load Excel and normalize column names
    excel_path = os.path.join(path_annotations, excel_name)
    if not os.path.exists(excel_path):
        raise FileNotFoundError(f"Excel not found at: {excel_path}")
    df = pd.read_excel(excel_path)
    df.rename(columns={c: c.strip().lower() for c in df.columns}, inplace=True)

    def find_col(pattern):
        for c in df.columns:
            if re.search(pattern, c, flags=re.I):
                return c
        raise KeyError(f"No column matches pattern: {pattern}")

    col_prefix = find_col(r"\bprefix\b")
    col_page   = find_col(r"\bpage\b")
    col_ods    = find_col(r"\bods")
    col_arg    = find_col(r"\bargument")

    # Filter by prefix
    msk = df[col_prefix].astype(str).str.startswith(prefix)
    df = df[msk].copy()
    if df.empty:
        raise ValueError(f"No rows in Excel match prefix '{prefix}'.")

    #2. Parse ODS list per row
    def parse_ods_list(x):
        if pd.isna(x):
            return []
        parts = re.split(r"[,\s]+", str(x).strip())
        out = []
        for p in parts:
            if p == "":
                continue
            try:
                out.append(int(p))
            except:
                pass
        return out

    #3. Build grouped dict {(ods, page): [arguments]} and flat list of all arguments
    grouped = defaultdict(list)
    grouped_by_goal = defaultdict(list)
    all_args = []

    for _, row in df.iterrows():
        try:
            page = int(pd.to_numeric(row[col_page], errors="coerce"))
        except Exception:
            continue
        if pd.isna(page):
            continue

        arg = str(row[col_arg]).strip()
        if not arg or arg.lower() == "nan":
            continue

        ods_list = parse_ods_list(row[col_ods])
        all_args.append(arg)
        for ods in ods_list:
            key = (int(ods), int(page))
            if keep_duplicates or (arg not in grouped[key]):
                grouped[key].append(arg)
            if keep_duplicates or (arg not in grouped_by_goal[int(ods)]):
                grouped_by_goal[int(ods)].append(arg)

    #4. Output files (.json per ODS and page, .txt for all arguments)
    records = []
    for (ods, page), args in sorted(grouped.items(), key=lambda x: (x[0][0], x[0][1])):
        records.append({
            "ods": int(ods),
            "page": int(page),
            "arguments": args,
            "source_file": f"{prefix}annotations.xlsx",
        })

    if json_basename is None:
        json_basename = f"{prefix}_AllArgs_annotations"
    if txt_basename is None:
        txt_basename = f"{prefix}_AllArgs_annotations_arguments"
    json_path = os.path.join(path_annotations, f"{json_basename}.json")
    txt_path  = os.path.join(path_annotations, f"{txt_basename}.txt")
    txt_bygoal_path = os.path.join(path_annotations, f"{prefix}_AllArgs_annotations_by_goal.txt")

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(records, f, indent=2, ensure_ascii=False)
    with open(txt_path, "w", encoding="utf-8") as f:
        for a in all_args:
            cleaned_a = str(a).replace('\n', ' ').strip()
            f.write(f"'{cleaned_a}',\n")
    with open(txt_bygoal_path, "w", encoding="utf-8") as f:
        json.dump(grouped_by_goal, f, indent=2, ensure_ascii=False)

    results = {"json_path": json_path, "txt_path": txt_path, "records_count": len(records), "args_count": len(all_args)}
    print(", ".join(f"{k}: {v}" for k, v in results.items()))
    return results


In [3]:
path_annotations  = "..\\Data\\Annotations\\"
prefix = "GLOBAL_SGD2023_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

prefix = "GLOBAL_SGD2024_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

prefix = "GLOBAL_SGD2025_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

prefix = "OCDE_2024_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

prefix = "G20_2024_"

make_annotations_outputs(
    path_annotations,
    prefix,
    excel_name= "TFM-ExtractedArgumentsAnnotations.xlsx",
    json_basename= None,
    txt_basename= None,
    keep_duplicates = True)

json_path: ..\Data\Annotations\GLOBAL_SGD2023__AllArgs_annotations.json, txt_path: ..\Data\Annotations\GLOBAL_SGD2023__AllArgs_annotations_arguments.txt, records_count: 229, args_count: 235
json_path: ..\Data\Annotations\GLOBAL_SGD2024__AllArgs_annotations.json, txt_path: ..\Data\Annotations\GLOBAL_SGD2024__AllArgs_annotations_arguments.txt, records_count: 198, args_count: 415
json_path: ..\Data\Annotations\GLOBAL_SGD2025__AllArgs_annotations.json, txt_path: ..\Data\Annotations\GLOBAL_SGD2025__AllArgs_annotations_arguments.txt, records_count: 149, args_count: 166
json_path: ..\Data\Annotations\OCDE_2024__AllArgs_annotations.json, txt_path: ..\Data\Annotations\OCDE_2024__AllArgs_annotations_arguments.txt, records_count: 151, args_count: 150
json_path: ..\Data\Annotations\G20_2024__AllArgs_annotations.json, txt_path: ..\Data\Annotations\G20_2024__AllArgs_annotations_arguments.txt, records_count: 51, args_count: 50


{'json_path': '..\\Data\\Annotations\\G20_2024__AllArgs_annotations.json',
 'txt_path': '..\\Data\\Annotations\\G20_2024__AllArgs_annotations_arguments.txt',
 'records_count': 51,
 'args_count': 50}

### 1. Define functions for Metric vs. Manual depuration

Definitions:
- Ground truth positives (gold) = total_args of the annotation row.
- True Positives (TP) = total_args_kept for each model.
- Predicted Positives = total_args for each model.

Metrics:
1. **Precision** = TP / Predicted Positives

In [28]:
def compute_prf(path_metrics_extracted, filename="AllArgs_overall_summary.csv"):
    path = os.path.join(path_metrics_extracted, filename)
    df = pd.read_csv(path)

    gold_total = int(df.loc[df["model"] == "annotation", "total_args"].iloc[0])

    rows = []
    for _, row in df.iterrows():
        model = row["model"]
        if model == "annotation":
            continue  

        tp = int(row["total_args_kept"])
        pred_total = int(row["total_args"])

        precision = tp / pred_total if pred_total > 0 else 0
        rows.append({
            "model": model,
            "total": gold_total,
            "extracted_total": pred_total,
            "real_argument": tp,
            "precision": round(precision, 4),
        })

    results = pd.DataFrame(rows)
    return results

# Example usage
path_metrics_extracted = "../Data/MetricsExtracted"



### 2. Define functions for Metric vs. Manual annotation


In [ ]:
# Load argument texts and goal-based JSON

def load_args_txt(path):
    args = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            s = line.strip().rstrip(",")
            s = re.sub(r"^'+|^\"+|'+$|\"+$", "", s).strip()
            if s:
                args.append(s)
    return args

def load_by_goal_json_txt(path):
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return {str(k): v for k, v in data.items()}

def build_path_overall(path_dir, prefix, model_name):
    fn = f"{prefix}__AllArgs_{model_name}_arguments.txt"
    return os.path.join(path_dir, fn)

In [ ]:
#### Text + embeddings

_EMBED = None
def get_embedder(model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
    global _EMBED
    if _EMBED is None:
        _EMBED = SentenceTransformer(model_name)
    return _EMBED

def normalize(s):
    s = re.sub(r"\s+", " ", s.strip())
    return s

def embed_texts(texts):
    emb = get_embedder().encode([normalize(t) for t in texts], batch_size=64, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    return emb

# Matching by similarity argument extracted with ground truth (minimizing Hungarian cost = 1 - cosine similarity)
def optimal_matching(refs, preds):

    if len(refs) == 0 or len(preds) == 0:
        return [], np.zeros((len(refs), len(preds)))
    E_ref = embed_texts(refs)
    E_pred = embed_texts(preds)
    S = cosine_similarity(E_ref, E_pred) 

    # Optimal search (min cost)
    cost = 1.0 - S
    r_idx, p_idx = linear_sum_assignment(cost)
    pairs = [(ri, pj, float(S[ri, pj])) for ri, pj in zip(r_idx, p_idx)]

    return pairs, S

# Pairwise metrics
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method3

def rougeL_f1(a, b):
    return _rouge.score(normalize(a), normalize(b))["rougeL"].fmeasure

def _tok_words(s: str):
    return _tok.tokenize(re.sub(r"\s+", " ", s.strip()))

def bleu_pair(a: str, b: str) -> float:
    ref = [_tok_words(a)]
    hyp = _tok_words(b)
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    smooth = SmoothingFunction().method3
    return sentence_bleu(ref, hyp, smoothing_function=smooth,
                         weights=(0.25, 0.25, 0.25, 0.25))


def berts_pairwise(a_list, b_list):
    P, R, F = bert_score(a_list, b_list, lang="en", rescale_with_baseline=True)
    return float(F.mean())

###################################
# Scoring one model vs ground truth

def evaluate_one(pred_args, gold_args, sim_threshold = 0.75,
                 use_bertscore = USE_BERTSCORE):
    
    pairs, S = optimal_matching(gold_args, pred_args) 

    # Best alignment stats
    sims = [s for _, _, s in pairs]
    avg_sim = float(np.mean(sims)) if sims else 0.0

    # Classification by similarity threshold
    matched = sum(s >= sim_threshold for s in sims) if sims else 0
    precision = matched / max(len(pred_args), 1)
    recall    = matched / max(len(gold_args), 1)
    f1        = 2*precision*recall / max((precision+recall), 1e-9)

    # ROUGE-L / BLEU over aligned pairs (macro average)
    rouge_vals, bleu_vals = [], []
    for gi, pj, _ in pairs:
        rouge_vals.append(rougeL_f1(gold_args[gi], pred_args[pj]))
        bleu_vals.append(bleu_pair(gold_args[gi], pred_args[pj]))
    rougeL_avg = float(np.mean(rouge_vals)) if rouge_vals else 0.0
    bleu_avg   = float(np.mean(bleu_vals)) if bleu_vals else 0.0

    # Optional BERTScore over aligned pairs
    bertscore_avg = 0.0
    if use_bertscore and pairs:
        a = [gold_args[gi] for gi, _, _ in pairs]
        b = [pred_args[pj] for _, pj, _ in pairs]
        bertscore_avg = berts_pairwise(a, b)

    return {
        "n_gold": len(gold_args),
        "n_pred": len(pred_args),
        "avg_pair_similarity": round(avg_sim, 4),
        "precision@thr": round(precision, 4),
        "recall@thr": round(recall, 4),
        "f1@thr": round(f1, 4),
        "rougeL_f1_avg": round(rougeL_avg, 4),
        "bleu_avg": round(bleu_avg, 4),
        **({"bertscore_f1_avg": round(bertscore_avg, 4)} if use_bertscore else {}),
    }

###################################
# Scoring by goal (precision, recall, f1)

def evaluate_by_goal(pred_by_goal, gold_by_goal, sim_threshold=0.75):
    
    goals = sorted(set(map(str, pred_by_goal.keys())) | set(map(str, gold_by_goal.keys())), key=lambda x:int(x))
    rows = []
    for g in goals:
        pred = pred_by_goal.get(g, [])
        gold = gold_by_goal.get(g, [])
        res  = evaluate_one(pred, gold, sim_threshold)
        rows.append({"goal": g, **res})
    df = pd.DataFrame(rows)

    # micro across goals
    micro = {
        "goal": "MICRO",
        "n_gold": int(df["n_gold"].sum()),
        "n_pred": int(df["n_pred"].sum()),
    }
    # recompute micro precision/recall/f1 from totals using matched counts
    # Rebuild matches to count true positives (>=thr)

    tp = 0
    for g in goals:
        pred = pred_by_goal.get(g, [])
        gold = gold_by_goal.get(g, [])
        pairs, _ = optimal_matching(gold, pred)
        tp += sum(s >= 0.75 for _, _, s in pairs)
    p = tp / max(int(df["n_pred"].sum()), 1)
    r = tp / max(int(df["n_gold"].sum()), 1)
    f1 = 2*p*r / max(p+r, 1e-9)
    micro.update({"precision@thr": round(p,4), "recall@thr": round(r,4), "f1@thr": round(f1,4)})

    return df, pd.DataFrame([micro])


In [30]:
############### Metric vs. Manual Annotation ############### 

def evaluate_models(path_args_extracted, path_gt_args, prefix, gold_model_name,
                    model_names, sim_threshold=0.75):

    def p_args(m):       return os.path.join(path_args_extracted, f"{prefix}_AllArgs_{m}_arguments.txt")
    def p_bygoal(m):     return os.path.join(path_args_extracted, f"{prefix}_AllArgs_{m}_by_goal.txt")
    def p_args_gt(m):    return os.path.join(path_gt_args,       f"{prefix}_AllArgs_{m}_arguments.txt")
    def p_bygoal_gt(m):  return os.path.join(path_gt_args,       f"{prefix}_AllArgs_{m}_by_goal.txt")

    # Embeddings
    emb_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    def encode(texts):
        if not texts:
            return np.zeros((0, emb_model.get_sentence_embedding_dimension()), dtype=np.float32)
        return emb_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)

    def cos_sim_mat(A, B):
        if A.size == 0 or B.size == 0:
            return np.zeros((A.shape[0], B.shape[0]), dtype=np.float32)
        return A @ B.T  # normalized -> cosine

    def greedy_match(sim, thr):
        if sim.size == 0: return []
        pairs, used_i, used_j = [], set(), set()
        idxs = np.dstack(np.unravel_index(np.argsort(sim, axis=None)[::-1], sim.shape))[0]
        for i, j in idxs:
            if sim[i, j] < thr: break
            if i in used_i or j in used_j: 
                continue
            used_i.add(int(i)); used_j.add(int(j))
            pairs.append((int(i), int(j), float(sim[i, j])))
        return pairs

    def jaccard_exact(pred, gold):
        sp, sg = set(map(normalize, pred)), set(map(normalize, gold))
        if not sp and not sg: return 1.0
        u = len(sp | sg)
        return len(sp & sg) / u if u else 0.0

    # Manual Annotations (gold)
    gold_all  = load_args_txt(p_args_gt(gold_model_name))
    gold_goal = load_by_goal_json_txt(p_bygoal_gt(gold_model_name))

    overall_rows, bygoal_frames = [], []

    #  Embeddings gold
    E_gold = encode(gold_all)
    gold_norm_set = set(map(normalize, gold_all))

    # collect predictions (strings + embeddings) for agreement
    preds_by_model = {}
    E_preds_by_model = {}

    iaa_rows = []

    for m in model_names:
        print(f"Evaluating model: {m}")
        pred_all  = load_args_txt(p_args(m))
        pred_goal = load_by_goal_json_txt(p_bygoal(m))

        # == OVERALL ==
        overall = evaluate_one(pred_all, gold_all, sim_threshold)
        E_pred_all = encode(pred_all)
        sim_all = cos_sim_mat(E_gold, E_pred_all)
        pairs_all = greedy_match(sim_all, sim_threshold)
        matched_all = len(pairs_all)
        n_gold_all = len(gold_all)
        n_pred_all = len(pred_all)
        coverage_ratio = (matched_all / n_gold_all) if n_gold_all else 0.0
        overgeneration = ((n_pred_all - matched_all) / n_pred_all) if n_pred_all else 0.0

        overall_rows.append({
            "model": m,
            **overall,
            "coverage_ratio": round(coverage_ratio, 4),
            "overgeneration": round(overgeneration, 4),
        })

        # == BY_GOAL ==
        df_by, df_micro = evaluate_by_goal(pred_goal, gold_goal, sim_threshold)
        cov_over_rows = []
        all_goals = sorted(set(map(int, gold_goal.keys())) | set(map(int, pred_goal.keys())))
        for g in all_goals:
            gold_g = gold_goal.get(str(g), []) if isinstance(next(iter(gold_goal.keys())), str) else gold_goal.get(g, [])
            pred_g = pred_goal.get(str(g), []) if isinstance(next(iter(pred_goal.keys())), str) else pred_goal.get(g, [])

            Eg = encode(gold_g)
            Ep = encode(pred_g)
            sim_g = cos_sim_mat(Eg, Ep)
            pairs_g = greedy_match(sim_g, sim_threshold)
            matched_g = len(pairs_g)
            n_gold_g = len(gold_g)
            n_pred_g = len(pred_g)
            cov_g = (matched_g / n_gold_g) if n_gold_g else 0.0
            over_g = ((n_pred_g - matched_g) / n_pred_g) if n_pred_g else 0.0
            cov_over_rows.append({"goal": g, "coverage_ratio": cov_g, "overgeneration": over_g})

        cov_over_df = pd.DataFrame(cov_over_rows)
        goal_col = None
        for cand in ["goal", "ods", "sdg"]:
            if cand in df_by.columns:
                goal_col = cand
                break
        if goal_col is None:
            goal_col = df_by.columns[0]

        def to_int_series(s):
            try:
                return pd.to_numeric(s, errors="raise").astype("Int64")
            except Exception:
                return None

        left_goal_int  = to_int_series(df_by[goal_col])
        right_goal_int = to_int_series(cov_over_df["goal"])

        if left_goal_int is not None and right_goal_int is not None:
            df_by = df_by.copy()
            df_by[goal_col] = left_goal_int
            cov_over_df = cov_over_df.copy()
            cov_over_df["goal"] = right_goal_int
        else:
            df_by = df_by.copy()
            df_by[goal_col] = df_by[goal_col].astype(str)
            cov_over_df = cov_over_df.copy()
            cov_over_df["goal"] = cov_over_df["goal"].astype(str)

        df_by = df_by.merge(cov_over_df.rename(columns={"goal": goal_col}),
                            on=goal_col, how="left")
        
        df_by["coverage_ratio"] = df_by["coverage_ratio"].fillna(0.0).round(4)
        df_by["overgeneration"] = df_by["overgeneration"].fillna(0.0).round(4)

        df_by.insert(0, "model", m)
        df_micro.insert(0, "model", m)
        bygoal_frames.append(df_by)
        bygoal_frames.append(df_micro)

        preds_by_model[m] = pred_all
        E_preds_by_model[m] = E_pred_all

        # == AGREEMENT PER MODEL ==
        jac_exact = jaccard_exact(pred_all, gold_all)
        inter = matched_all
        union = len(gold_all) + len(pred_all) - inter
        fuzzy_j = inter / union if union else 0.0

        iaa_rows.append({
            "model": m,
            "jaccard_vs_gold": round(jac_exact, 4),
            "fuzzy_jaccard_vs_gold": round(fuzzy_j, 4),
            "n_gold": len(gold_all),
            "n_pred": len(pred_all),
            "matches_at_thr": inter
        })

    iaa_per_model = pd.DataFrame(iaa_rows).sort_values("fuzzy_jaccard_vs_gold", ascending=False).reset_index(drop=True)

    # == AGREEMENT OVERALL ==
    n_gold = len(gold_all)
    counts_per_gold = np.zeros(n_gold, dtype=int)
    for m in model_names:
        E_pred = E_preds_by_model[m]
        sim = cos_sim_mat(E_gold, E_pred)
        pairs = greedy_match(sim, sim_threshold)
        hit_idxs = {gi for gi, _, _ in pairs}
        for gi in hit_idxs:
            counts_per_gold[gi] += 1

    cov_ge1 = (counts_per_gold >= 1).mean() if n_gold else 0.0
    cov_ge2 = (counts_per_gold >= 2).mean() if n_gold else 0.0
    cov_ge3 = (counts_per_gold >= 3).mean() if n_gold else 0.0
    cov_ge5 = (counts_per_gold >= 5).mean() if n_gold else 0.0

    total_preds, total_supported = 0, 0
    for m in model_names:
        E_self = E_preds_by_model[m]
        total_preds += E_self.shape[0]
        others = [E_preds_by_model[mm] for mm in model_names if mm != m and E_preds_by_model[mm].shape[0] > 0]
        if not others or E_self.shape[0] == 0:
            continue
        E_union = np.vstack(others)
        sim = cos_sim_mat(E_self, E_union)
        total_supported += int((sim.max(axis=1) >= sim_threshold).sum()) if sim.size else 0
    pct_model_supported = (total_supported / total_preds) if total_preds else 0.0

    def fleiss_kappa_binary(present_matrix: np.ndarray) -> float:
        n_items, n_raters = present_matrix.shape
        if n_items == 0 or n_raters < 2: return 0.0
        p_i = present_matrix.mean(axis=1)
        Pbar = np.mean(p_i**2 + (1 - p_i)**2)
        p_bar = present_matrix.mean()
        Pbar_e = p_bar**2 + (1 - p_bar)**2
        denom = 1.0 - Pbar_e
        return (Pbar - Pbar_e) / denom if denom > 0 else 0.0

    if n_gold and model_names:
        present = np.zeros((n_gold, len(model_names)), dtype=int)
        for j, m in enumerate(model_names):
            sim = cos_sim_mat(E_gold, E_preds_by_model[m])
            pairs = greedy_match(sim, sim_threshold)
            if pairs:
                present[list({gi for gi, _, _ in pairs}), j] = 1
        fleiss_kappa_all = float(fleiss_kappa_binary(present))
    else:
        fleiss_kappa_all = 0.0

    iaa_overall = pd.DataFrame([{
        "models_compared": len(model_names),
        "gold_args": n_gold,
        "coverage_ge1": round(cov_ge1, 4),
        "coverage_ge2": round(cov_ge2, 4),
        "coverage_ge3": round(cov_ge3, 4),
        "coverage_ge5": round(cov_ge5, 4),
        "pct_model_args_supported_by_another": round(pct_model_supported, 4),
        "fleiss_kappa_all": round(fleiss_kappa_all, 4),
    }])

    return {
        "overall": pd.DataFrame(overall_rows).sort_values("f1@thr", ascending=False).reset_index(drop=True),
        "by_goal": pd.concat(bygoal_frames, ignore_index=True),
        "iaa_per_model": iaa_per_model,
        "iaa_overall": iaa_overall
    }


#### GLOBAL 2023 No keywords

In [32]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments No Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments No Keywords\\Stats"
prefix   = "GLOBAL_SGD2023_"
gt     = "annotations"        
models   = ["qwen2.5-3b","gemma3-4b","gemma3-27b", "llama3.3-70b", "deepseek-r1-70b"]

### 2. Retrieval metrics vs. manual depuration
print("Extraction quality metrics vs. manual depuration")

results_md = compute_prf(path_output_metrics)
display(results_md)

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)

Extraction quality metrics vs. manual depuration


,model,total,extracted_total,real_argument,precision
0,qwen2.5-3b,250,251,161,0.6414
1,gemma3-4b,250,4093,706,0.1725
2,gemma3-27b,250,708,330,0.4661
3,llama3.3-70b,250,817,477,0.5838
4,deepseek-r1-70b,250,1178,553,0.4694


Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: llama3.3-70b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,coverage_ratio,overgeneration
0,deepseek-r1-70b,234,553,0.8617,0.3038,0.7179,0.4269,0.7164,0.6252,0.7179,0.6962
1,gemma3-27b,234,334,0.7468,0.3323,0.4744,0.3908,0.5218,0.4235,0.4786,0.6647
2,llama3.3-70b,234,477,0.7947,0.2662,0.5427,0.3572,0.5912,0.5011,0.5427,0.7338
3,gemma3-4b,234,729,0.8217,0.2003,0.6239,0.3032,0.6443,0.5370,0.6239,0.7997
4,qwen2.5-3b,234,165,0.6499,0.2545,0.1795,0.2105,0.3429,0.2235,0.1838,0.7394



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,coverage_ratio,overgeneration
0,qwen2.5-3b,0,34,32,0.5656,0.1250,0.1176,0.1212,0.2321,0.0988,0.1176,0.8750
1,qwen2.5-3b,1,18,9,0.4593,0.0000,0.0000,0.0000,0.1291,0.0306,0.0000,1.0000
2,qwen2.5-3b,2,10,2,0.4726,0.0000,0.0000,0.0000,0.1374,0.0122,0.0000,1.0000
3,qwen2.5-3b,3,18,12,0.4926,0.0833,0.0556,0.0667,0.1408,0.0158,0.0556,0.9167
4,qwen2.5-3b,4,29,5,0.5565,0.2000,0.0345,0.0588,0.3043,0.2140,0.0345,0.8000
...,...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,20,10,0.8082,0.6000,0.3000,0.4000,0.5628,0.4608,0.3000,0.4000
91,deepseek-r1-70b,15,15,20,0.6725,0.3000,0.4000,0.3429,0.4486,0.3720,0.4000,0.7000
92,deepseek-r1-70b,16,52,29,0.6195,0.2759,0.1538,0.1975,0.3569,0.2646,0.1538,0.7241
93,deepseek-r1-70b,17,49,77,0.7428,0.2987,0.4694,0.3651,0.4880,0.3718,0.4694,0.7013



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,deepseek-r1-70b,0.1338,0.2714,234,553,168
1,gemma3-27b,0.1281,0.2456,234,334,112
2,llama3.3-70b,0.1208,0.2175,234,477,127
3,gemma3-4b,0.0995,0.1787,234,729,146
4,qwen2.5-3b,0.0559,0.1208,234,165,43



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all
0,5,234,0.9231,0.735,0.5171,0.0812,0.8663,0.3187


#### GLOBAL 2023 keywords

In [34]:
path_gt_args = "..\\Data\\Annotations"

path_args_extracted = "..\\Data\\Processed Arguments Keywords\\Args"
path_output_metrics = "..\\Data\\Processed Arguments Keywords\\Stats"
prefix   = "GLOBAL_SGD2023_"
gt     = "annotations"        
models   = ["llama3.3-70b","qwen2.5-3b","gemma3-4b","gemma3-27b", "deepseek-r1-70b"]

### 2. Retrieval metrics vs. manual depuration
print("Extraction quality metrics vs. manual depuration")

results_md = compute_prf(path_output_metrics)
display(results_md)

### 1. Quality metrics vs. manual annotation
results = evaluate_models(path_args_extracted, path_gt_args,
                          prefix, gt, models, sim_threshold=0.75)

print("Extraction quality metrics vs. manual annotation")
for k, df in results.items():
    print(f"\n== {k.upper()} ==")
    display(df)

# Save the results
results["overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_overall.csv"), index=False)
results["by_goal"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_byGoal.csv"), index=False)
results["iaa_per_model"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_per_model.csv"), index=False)
results["iaa_overall"].to_csv(os.path.join(path_output_metrics, f"{prefix}_ExtractionStats_iaa_overall.csv"), index=False)



Extraction quality metrics vs. manual depuration


,model,total,extracted_total,real_argument,precision
0,qwen2.5-3b,250,153,95,0.6209
1,gemma3-4b,250,3195,396,0.1239
2,gemma3-27b,250,697,284,0.4075
3,llama3.3-70b,250,859,359,0.4179
4,deepseek-r1-70b,250,1148,368,0.3206


Evaluating model: llama3.3-70b
Evaluating model: qwen2.5-3b
Evaluating model: gemma3-4b
Evaluating model: gemma3-27b
Evaluating model: deepseek-r1-70b
Extraction quality metrics vs. manual annotation

== OVERALL ==


,model,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,coverage_ratio,overgeneration
0,deepseek-r1-70b,234,368,0.8038,0.3723,0.5855,0.4551,0.6066,0.5054,0.5812,0.6304
1,gemma3-27b,234,284,0.7286,0.3873,0.4701,0.4247,0.5061,0.3969,0.4744,0.6092
2,llama3.3-70b,234,359,0.7682,0.3315,0.5085,0.4013,0.5526,0.4522,0.5085,0.6685
3,gemma3-4b,234,396,0.7524,0.2828,0.4786,0.3556,0.5133,0.3837,0.4829,0.7146
4,qwen2.5-3b,234,96,0.6984,0.3229,0.1325,0.1879,0.3718,0.2353,0.1410,0.6562



== BY_GOAL ==


,model,goal,n_gold,n_pred,avg_pair_similarity,precision@thr,recall@thr,f1@thr,rougeL_f1_avg,bleu_avg,coverage_ratio,overgeneration
0,llama3.3-70b,0,34,75,0.7328,0.1733,0.3824,0.2385,0.4523,0.3217,0.3824,0.8267
1,llama3.3-70b,1,18,11,0.6378,0.2727,0.1667,0.2069,0.3966,0.2939,0.1667,0.7273
2,llama3.3-70b,2,10,7,0.7085,0.4286,0.3000,0.3529,0.4853,0.4224,0.3000,0.5714
3,llama3.3-70b,3,18,15,0.5922,0.3333,0.2778,0.3030,0.3439,0.2145,0.2778,0.6667
4,llama3.3-70b,4,29,20,0.6381,0.4000,0.2759,0.3265,0.4293,0.2969,0.2759,0.6000
...,...,...,...,...,...,...,...,...,...,...,...,...
90,deepseek-r1-70b,14,20,5,0.8912,0.8000,0.2000,0.3200,0.7455,0.6677,0.2000,0.2000
91,deepseek-r1-70b,15,15,21,0.6367,0.1905,0.2667,0.2222,0.3614,0.2810,0.2667,0.8095
92,deepseek-r1-70b,16,52,27,0.6754,0.2593,0.1346,0.1772,0.3722,0.2591,0.1346,0.7407
93,deepseek-r1-70b,17,49,44,0.6969,0.3864,0.3469,0.3656,0.4347,0.3174,0.3469,0.6136



== IAA_PER_MODEL ==


,model,jaccard_vs_gold,fuzzy_jaccard_vs_gold,n_gold,n_pred,matches_at_thr
0,deepseek-r1-70b,0.1214,0.2918,234,368,136
1,gemma3-27b,0.1087,0.2727,234,284,111
2,llama3.3-70b,0.1295,0.2511,234,359,119
3,gemma3-4b,0.0833,0.2186,234,396,113
4,qwen2.5-3b,0.0259,0.1111,234,96,33



== IAA_OVERALL ==


,models_compared,gold_args,coverage_ge1,coverage_ge2,coverage_ge3,coverage_ge5,pct_model_args_supported_by_another,fleiss_kappa_all
0,5,234,0.8291,0.6368,0.4316,0.0513,0.8556,0.3652
